<!-- cspell:ignore atleast dalitz fromkeys lightgray mystnb pyodide xscale yscale zorder -->

# Phase space generators

:::{autolink-concat}
:::

Amplitude analysis needs samples of four-momenta that are distributed uniformly over the phase space of a decay. In TensorWaves, such a sample comes from a {class}`.DataGenerator`. The recommended generator is {class}`.EvtGenPhaseSpaceGenerator`, which uses [EvtGen](https://gitlab.cern.ch/evtgen/evtgen) through [`pyevtgen`](https://github.com/redeboer/pyevtgen-test) and which all examples in this documentation use. Phase space samples can also be generated with {class}`.PhaseSpaceGenerator`, which uses [`phasespace`](https://phasespace.readthedocs.io) with NumPy, JAX, or TensorFlow as computational backend. This page compares the performance and the distributions of these generators. All of them take the same masses and return a {obj}`.DataSample` of NumPy arrays with the same structure, so they can be swapped without changing the rest of an analysis.

In [ ]:
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib_inline.backend_inline import set_matplotlib_formats

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
set_matplotlib_formats("svg")

## Available generators

{class}`.PhaseSpaceGenerator` uses NumPy by default and does not require TensorFlow. Its backend is selected with the `backend` argument. Each generator below gets a {class}`.RealNumberGenerator` that it can use directly where possible. The function below is the single place that defines which generators this page compares. All tables and figures on this page loop over the generators it returns.

In [ ]:
from typing import Literal

from tensorwaves.data import (
    EvtGenPhaseSpaceGenerator,
    NumpyUniformRNG,
    PhaseSpaceGenerator,
    TFUniformRealNumberGenerator,
)


def create_phasespace_generator(
    initial_state_mass: float,
    final_state_masses: dict[int, float],
    backend: Literal["numpy", "tensorflow", "jax"],
) -> PhaseSpaceGenerator:
    generator = PhaseSpaceGenerator(
        initial_state_mass, final_state_masses, backend=backend
    )
    generator.show_progress = False
    return generator


def create_generators(initial_state_mass: float, final_state_masses: dict[int, float]):
    masses = initial_state_mass, final_state_masses
    return {
        "EvtGen": (EvtGenPhaseSpaceGenerator(*masses), NumpyUniformRNG(seed=0)),
        "phasespace (NumPy)": (
            create_phasespace_generator(*masses, backend="numpy"),
            NumpyUniformRNG(seed=0),
        ),
        "phasespace (JAX)": (
            create_phasespace_generator(*masses, backend="jax"),
            NumpyUniformRNG(seed=0),
        ),
        "phasespace (TensorFlow)": (
            create_phasespace_generator(*masses, backend="tensorflow"),
            TFUniformRealNumberGenerator(seed=0),
        ),
    }

| Generator | Package | Extra | Events | Use of the {class}`.RealNumberGenerator` |
| --- | --- | --- | --- | --- |
| {class}`.EvtGenPhaseSpaceGenerator` | `pyevtgen` | `evtgen` | unweighted | any type only provides a seed |
| {class}`.PhaseSpaceGenerator` with `backend="numpy"` (default) | `phasespace` | `phsp` | unweighted through hit-and-miss | {class}`.NumpyUniformRNG` is used directly, any other type only provides a seed |
| {class}`.PhaseSpaceGenerator` with `backend="jax"` | `phasespace` | `phsp,jax` | unweighted through hit-and-miss | any type only provides a seed |
| {class}`.PhaseSpaceGenerator` with `backend="tensorflow"` | `phasespace` | `phsp,tf` | unweighted through hit-and-miss | {class}`.TFUniformRealNumberGenerator` is used directly, any other type only provides a seed |

{class}`.WeightedPhaseSpaceGenerator` takes the same `backend` argument and returns weighted events instead. The installation of these extras is described on {doc}`/install`.

## Performance

We measure how long each generator takes for decays of the $J/\psi$ into three, four, and five pions and for several sample sizes. TensorFlow and JAX compile functions the first time they are called and `phasespace` compiles them again after each change of backend, so we first call each generator once without timing it. After that, we take the fastest of three calls, which limits the influence of other processes on the machine. These timings only give an impression for these decays on the machine that builds this page.

In [ ]:
def measure(generator, rng, size: int) -> float:
    start = time.perf_counter()
    generator.generate(size, rng)
    return time.perf_counter() - start


j_psi_mass = 3.0969
pion_mass = 0.1396
sample_sizes = [1_000, 10_000, 100_000]
n_repeats = 3
benchmark = {}
for n_particles in [3, 4, 5]:
    pions = dict.fromkeys(range(n_particles), pion_mass)
    for name, (generator, rng) in create_generators(j_psi_mass, pions).items():
        measure(generator, rng, sample_sizes[0])
        benchmark[n_particles, name] = [
            min(measure(generator, rng, size) for _ in range(n_repeats))
            for size in sample_sizes
        ]

In [ ]:
df_benchmark = pd.DataFrame(benchmark, index=sample_sizes).T
df_benchmark.index.names = ["Final state particles", "Generator"]
df_benchmark.columns = [f"{size:,} events [s]" for size in sample_sizes]
df_benchmark.style.format("{:.4f}")

In [ ]:
n_particle_values = sorted({n for n, _ in benchmark})
fig, axes = plt.subplots(
    ncols=len(n_particle_values),
    figsize=(4 * len(n_particle_values), 3.5),
    sharey=True,
)
for ax, n_particles in zip(axes, n_particle_values, strict=True):
    for (n, name), durations in benchmark.items():
        if n == n_particles:
            ax.plot(sample_sizes, durations, marker="o", label=name)
    ax.set_title(f"{n_particles}-body decay")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("Number of events")
axes[0].set_ylabel("Time [s]")
axes[0].legend(fontsize="small")
fig.tight_layout()
plt.show()

EvtGen is the fastest generator for all of these decays, and its advantage grows with the number of final state particles. The `phasespace` backends become slower with each additional particle, partly because a smaller fraction of the weighted events survive the hit-and-miss step.

## Comparing distributions

We now generate a larger sample for a three-body decay with different final state masses. The events of different generators are not identical, not even with the same seed, but they should follow the same distribution.

In [ ]:
generators = create_generators(
    initial_state_mass=3.0969,
    final_state_masses={0: 0.1350, 1: 0.1396, 2: 0.4937},
)
n_events = 500_000
samples = {
    name: generator.generate(n_events, rng)
    for name, (generator, rng) in generators.items()
}

In [ ]:
pd.DataFrame(
    {
        "Array type": [type(sample["p0"]).__name__ for sample in samples.values()],
        "Final state": [sorted(sample) for sample in samples.values()],
        "Shape": [sample["p0"].shape for sample in samples.values()],
    },
    index=list(samples),
)

A uniform phase space sample shows up as a flat Dalitz plot within the kinematic boundary.

In [ ]:
def invariant_mass_squared(p: np.ndarray) -> np.ndarray:
    return p[:, 0] ** 2 - np.sum(p[:, 1:] ** 2, axis=1)


fig, axes = plt.subplots(
    ncols=len(samples), figsize=(3 * len(samples), 3), sharex=True, sharey=True
)
for ax, (name, sample) in zip(np.atleast_1d(axes), samples.items(), strict=True):
    s01 = invariant_mass_squared(sample["p0"] + sample["p1"])
    s12 = invariant_mass_squared(sample["p1"] + sample["p2"])
    ax.hist2d(s01, s12, bins=60, cmin=1, rasterized=True)
    ax.set_title(name, fontsize="small")
    ax.set_xlabel("$m_{01}^2$ [GeV$^2$]")
np.atleast_1d(axes)[0].set_ylabel("$m_{12}^2$ [GeV$^2$]")
fig.tight_layout()
plt.show()

The invariant mass projections should agree between generators. The lower panels show the ratio of each generator to EvtGen, with the statistical uncertainty of the EvtGen sample as a band.

In [ ]:
pairs = {"m_{01}": ("p0", "p1"), "m_{02}": ("p0", "p2"), "m_{12}": ("p1", "p2")}
reference = next(iter(samples))
histograms = {}
fig, axes = plt.subplots(
    nrows=2,
    ncols=len(pairs),
    figsize=(4 * len(pairs), 4.5),
    height_ratios=[3, 1],
    sharex="col",
)
for (top, bottom), (label, (a, b)) in zip(axes.T, pairs.items(), strict=True):
    masses = {
        name: np.sqrt(invariant_mass_squared(sample[a] + sample[b]))
        for name, sample in samples.items()
    }
    edges = np.histogram_bin_edges(masses[reference], bins=60)
    counts = {name: np.histogram(m, bins=edges)[0] for name, m in masses.items()}
    histograms[label] = counts
    reference_counts = np.where(counts[reference] > 0, counts[reference], np.nan)
    for name, n in counts.items():
        top.stairs(n, edges, label=name)
        if name != reference:
            bottom.stairs(n / reference_counts, edges, label=name)
    bottom.stairs(
        1 + 1 / np.sqrt(reference_counts),
        edges,
        baseline=1 - 1 / np.sqrt(reference_counts),
        fill=True,
        color="lightgray",
        zorder=0,
    )
    bottom.set_ylim(0.9, 1.1)
    bottom.set_xlabel(f"${label}$ [GeV]")
axes[0, 0].set_ylabel("Events")
axes[1, 0].set_ylabel("Ratio")
axes[0, 0].legend(fontsize="small")
fig.tight_layout()
plt.show()

To quantify the agreement, we compute a two-sample $\chi^2$ per degree of freedom between each generator and EvtGen, using bins with sufficient statistics. Values around one mean that the samples are statistically compatible.

In [ ]:
def chi2_per_dof(n1: np.ndarray, n2: np.ndarray) -> float:
    mask = n1 + n2 > 50
    return np.sum((n1 - n2)[mask] ** 2 / (n1 + n2)[mask]) / np.count_nonzero(mask)


pd.DataFrame({
    f"${label}$": {
        name: chi2_per_dof(n, counts[reference])
        for name, n in counts.items()
        if name != reference
    }
    for label, counts in histograms.items()
}).round(2)

## Which generator to choose

{class}`.EvtGenPhaseSpaceGenerator` is the default choice: it is the fastest generator and needs neither TensorFlow nor JAX. It works with any {class}`.RealNumberGenerator`, which only provides a seed for the random engine of EvtGen. There are two limitations:

- It provides no weights, because EvtGen applies accept-reject sampling internally. The {class}`.IntensityDistributionGenerator` does not need them, as the {doc}`/amplitude-analysis` page shows.
- Its accept-reject efficiency depends on the masses of the final state. Generation can become impractically slow for many final state particles, and a running call cannot be interrupted.

{class}`.PhaseSpaceGenerator` and {class}`.WeightedPhaseSpaceGenerator` are useful where `pyevtgen` is not available. With the NumPy backend, `phasespace` only depends on NumPy, so these generators may also work where neither `pyevtgen`, TensorFlow, nor JAX can be installed, for instance in the browser through [Pyodide](https://pyodide.org). This has not been tested. TensorFlow reproduces samples from earlier TensorWaves versions, when combined with a {class}`.TFUniformRealNumberGenerator`. {class}`.WeightedPhaseSpaceGenerator` provides event weights. There are two things to keep in mind with these backends:

- The same seed gives a different sample on each backend. Samples are only reproducible within one backend.
- `phasespace` selects its backend per process, and a generator activates its backend each time it generates. Each switch makes `phasespace` recompile, so alternating between backends is slower than the timings above suggest, and generators with different backends should not be used concurrently in threads.